# Where does the Hough lane detector fail? (CULane)

1. A headless version of the classic Canny + Hough detector: one image and a fixed box in, two lines out.
2. Each test image is scored against the ground-truth lines of your own lane.
3. A table with one row per image: scene features, error, pass/fail.
4. A random forest predicts pass/fail from the scene features, to show which ones matter.
5. Charts.

Add a CULane dataset to the notebook, then run all cells.

In [ ]:
import os
import random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split

SEARCH_IN = "/kaggle/input"  # the CULane folders are searched for in here
MAX_PER_CATEGORY = 1000      # images sampled per category; None = all of them
SEED = 0

PASS_PX = 25         # an image passes if its worse line is within this many pixels
ERROR_CAP = 100      # a missing line, or one further off than this, counts as this
SCORE_HEIGHTS = 8    # heights at which a line is compared with the true one

ROI_EXPANSION = 0.6     # each box is this much wider than the lane it was built from
# The image row where the car's hood starts, per camera: the box stops there.
# These three were read off sample images by eye. A camera not listed here
# gets an automatic estimate, which is less reliable.
HOOD_Y = {
    "driver_100_30frame": 440,
    "driver_193_90frame": 400,
    "driver_37_30frame": 398,
}
VISIBLE_CONTRAST = 15   # a true line this much brighter than the road counts as visible
STRAIGHT_LINES = True   # False lets a lane bend when the road curves

## 1. The headless detector

`detect_lanes(image, box)` is the first script with the windows, clicking, video and
frame-to-frame smoothing removed. `box` is four corners: bottom-left, top-left,
top-right, bottom-right. Each returned line is `[a, b, c]` for `x = a*y**2 + b*y + c`,
or `None` if no line was found on that side.

In [ ]:
def lane_edges(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    return cv2.Canny(blurred, 50, 150)


def box_mask(shape, box):
    mask = np.zeros(shape[:2], dtype=np.uint8)
    cv2.fillPoly(mask, [np.asarray(box, dtype=np.int32)], 255)
    return mask


def find_lane_pixels(binary, box):
    """Pick the edge pixels of the lane line nearest the box centre on each side.

    Find straight pieces of line, keep the ones slanted like lane lines, take
    the group closest to the middle on each side, then collect the edge
    pixels along that line.
    """
    height, width = binary.shape
    y_bottom = int(max(box[0, 1], box[3, 1]))
    center = (box[0, 0] + box[3, 0]) / 2
    nonzero_y, nonzero_x = binary.nonzero()
    empty = np.array([], dtype=np.int32)

    segments = cv2.HoughLinesP(
        binary, 1, np.pi / 180, threshold=15, minLineLength=15, maxLineGap=25
    )
    if segments is None:
        return empty, empty, empty, empty

    candidates = {"left": [], "right": []}
    for x1, y1, x2, y2 in segments.reshape(-1, 4).astype(float):
        if y1 == y2:
            continue
        dx_dy = (x2 - x1) / (y2 - y1)
        if abs(dx_dy) > 3.0:
            continue  # too flat to be our lane: hood edge, lights, far lines
        x_at_bottom = x1 + (y_bottom - y1) * dx_dy
        # Left lane lines lean right going up the image, right ones lean left.
        side = "left" if dx_dy < 0 else "right"
        if (side == "left") != (x_at_bottom < center):
            continue
        length = np.hypot(x2 - x1, y2 - y1)
        candidates[side].append((x_at_bottom, length, (x1, y1, x2, y2)))

    group_gap = max(20, width * 0.04)
    min_length = max(30, width * 0.04)
    near_line = max(12, width * 0.02)
    result = []
    for side in ("left", "right"):
        nearest_first = sorted(candidates[side], key=lambda c: abs(c[0] - center))
        chosen = None
        for x_at_bottom, _, _ in nearest_first:
            group = [c for c in nearest_first if abs(c[0] - x_at_bottom) < group_gap]
            if sum(c[1] for c in group) >= min_length:
                chosen = group
                break
        if chosen is None:
            result += [empty, empty]
            continue

        ys = np.concatenate([[c[2][1], c[2][3]] for c in chosen])
        xs = np.concatenate([[c[2][0], c[2][2]] for c in chosen])
        line = np.polyfit(ys, xs, 1)  # x as a function of y
        for _ in range(2):  # refine the line on the edge pixels near it
            near = np.abs(nonzero_x - np.polyval(line, nonzero_y)) < near_line
            if near.sum() >= 10 and np.ptp(nonzero_y[near]) > 5:
                line = np.polyfit(nonzero_y[near], nonzero_x[near], 1)
        near = np.abs(nonzero_x - np.polyval(line, nonzero_y)) < near_line
        result += [nonzero_x[near], nonzero_y[near]]

    return tuple(result)


def fit_curve(x_values, y_values, box_height):
    """A line needs enough pixels, spread over enough of the box's height."""
    if len(x_values) < max(30, box_height * 0.25):
        return None
    if len(np.unique(y_values)) < max(10, box_height * 0.08):
        return None
    if np.ptp(y_values) < box_height * 0.4:
        return None

    linear = np.polyfit(y_values, x_values, 1)
    if STRAIGHT_LINES:
        return np.array([0.0, linear[0], linear[1]])

    quadratic = np.polyfit(y_values, x_values, 2)
    linear_error = np.sqrt(np.mean((x_values - np.polyval(linear, y_values)) ** 2))
    curve_error = np.sqrt(np.mean((x_values - np.polyval(quadratic, y_values)) ** 2))
    sample_y = np.linspace(y_values.min(), y_values.max(), 50)
    difference = np.max(np.abs(
        np.polyval(quadratic, sample_y) - np.polyval(linear, sample_y)
    ))
    if curve_error < linear_error * 0.75 and difference > 8:
        return quadratic
    return np.array([0.0, linear[0], linear[1]])


def detect_lanes(image, box):
    """One BGR image and a fixed box in, (left_line, right_line) out."""
    box = np.asarray(box, dtype=np.int32)
    edges = cv2.bitwise_and(lane_edges(image), box_mask(image.shape, box))
    left_x, left_y, right_x, right_y = find_lane_pixels(edges, box)
    box_height = np.ptp(box[:, 1])
    return fit_curve(left_x, left_y, box_height), fit_curve(right_x, right_y, box_height)

## 2. The dataset

The test set comes with one list of images per category (`list/test_split/test0_normal.txt`
and so on). Each image has a `.lines.txt` label file: one lane per row, written as
`x y x y ...`. The two lanes nearest the middle of the image are your lane's lines.

In [ ]:
def find_folder(start, wanted, max_depth=6):
    """Look for a folder by name without walking through every image folder."""
    start = Path(start)
    for current, folders, _ in os.walk(start):
        if wanted in folders:
            return Path(current) / wanted
        depth = len(Path(current).relative_to(start).parts)
        folders[:] = [] if depth >= max_depth else [
            f for f in folders if not f.upper().endswith(".MP4")
        ]
    return None


split_folder = find_folder(SEARCH_IN, "test_split")
if split_folder is None:
    raise FileNotFoundError(f"No CULane 'list/test_split' folder under {SEARCH_IN}")

chooser = random.Random(SEED)
rows = []
for list_file in sorted(split_folder.glob("test*_*.txt")):
    category = list_file.stem.split("_", 1)[1]
    paths = [line.strip().lstrip("/") for line in list_file.read_text().splitlines()]
    paths = [path for path in paths if path]
    if MAX_PER_CATEGORY and len(paths) > MAX_PER_CATEGORY:
        paths = chooser.sample(paths, MAX_PER_CATEGORY)
    rows += [(path, category, path.split("/")[0]) for path in paths]
images = pd.DataFrame(rows, columns=["image", "category", "camera"])

def find_roots(start, cameras, max_depth=6):
    """Every folder that directly contains one of the camera folders.

    Some Kaggle copies nest a folder inside one of the same name
    (driver_100_30frame/driver_100_30frame/...), so camera folders are
    searched inside too.
    """
    start, roots = Path(start), []
    for current, folders, _ in os.walk(start):
        if cameras & set(folders):
            roots.append(Path(current))
        depth = len(Path(current).relative_to(start).parts)
        folders[:] = [] if depth >= max_depth else [
            f for f in folders if not f.upper().endswith(".MP4")
        ]
    return roots


ROOTS = find_roots(SEARCH_IN, set(images["camera"]))
print("Lists:", split_folder)
print("Camera folders found in:", [str(root) for root in ROOTS])


def locate(name):
    """Full path of a listed file, wherever in the dataset it actually is."""
    for root in ROOTS:
        if (root / name).exists():
            return root / name
    return None


# Images and labels can sit in different places in a Kaggle copy, and some
# copies are incomplete, so check which listed files really exist.
IMAGE_FILE = {name: locate(name) for name in images["image"].unique()}
LABEL_FILE = {
    name: locate(str(Path(name).with_suffix(".lines.txt"))) for name in IMAGE_FILE
}
images["has_image"] = images["image"].map(lambda name: IMAGE_FILE[name] is not None)
images["has_label"] = images["image"].map(lambda name: LABEL_FILE[name] is not None)
found = images.groupby("camera").agg(
    listed=("image", "size"), images_found=("has_image", "sum"),
    labels_found=("has_label", "sum"),
)
print(found)

for camera in found.index[(found["images_found"] == 0) | (found["labels_found"] == 0)]:
    wanted = images.loc[images["camera"] == camera, "image"].iloc[0]
    print(f"\nNothing usable for {camera}. The list asks for: {wanted}")
    for root in ROOTS:
        video_folder = (root / wanted).parent
        if (root / camera).exists():
            print("  in", root / camera, ":", sorted(os.listdir(root / camera))[:5])
        if video_folder.exists():
            print("  in", video_folder, ":", sorted(os.listdir(video_folder))[:6])

images = images[images["has_image"] & images["has_label"]].drop(
    columns=["has_image", "has_label"]
).reset_index(drop=True)
if images.empty:
    raise FileNotFoundError(
        "None of the listed test images were found together with a .lines.txt label. "
        "See the folder listings printed above."
    )
print(f"\nUsing {len(images)} images that have both the picture and its label.")
print(images.groupby(["camera", "category"]).size().unstack(fill_value=0))


def read_lanes(image_name):
    """The labelled lanes of one image, each an array of (x, y) sorted by y."""
    label_path = LABEL_FILE[image_name]
    lanes = []
    for row in label_path.read_text().splitlines():
        numbers = np.array(row.split(), dtype=float)
        points = numbers[: len(numbers) // 2 * 2].reshape(-1, 2)
        points = points[points[:, 0] >= 0]
        if len(points) >= 2:
            lanes.append(points[np.argsort(points[:, 1])])
    return lanes


def own_lane(lanes, width, height):
    """The lane nearest the middle on each side: {"left": points, "right": points}."""
    best = {"left": None, "right": None}
    best_x = {"left": -np.inf, "right": np.inf}
    for lane in lanes:
        lowest = lane[-6:]  # the part nearest the car
        if np.ptp(lowest[:, 1]) == 0:
            continue
        x_bottom = np.polyval(np.polyfit(lowest[:, 1], lowest[:, 0], 1), height - 1)
        if x_bottom < width / 2 and x_bottom > best_x["left"]:
            best["left"], best_x["left"] = lane, x_bottom
        elif x_bottom >= width / 2 and x_bottom < best_x["right"]:
            best["right"], best_x["right"] = lane, x_bottom
    return best

## 3. One fixed box per camera folder

Instead of clicking, each camera's box is worked out once and reused for every image
from that camera:

- **Sides:** the typical position of your lane's two lines in up to 300 labels, widened.
- **Bottom:** the top of the car's hood, from `HOOD_Y` in the first cell.
- **Top:** 20% of the way down from where the two lines meet.

Check the pictures below: the box should cover road only, ending at the hood.

In [ ]:
def load(image_name):
    return cv2.imread(str(IMAGE_FILE[image_name]))


def average_image(image_names, limit=150):
    total, count = None, 0
    for image in map(load, list(image_names)[:limit]):
        if image is None or (total is not None and image.shape != total.shape):
            continue
        total = image.astype(np.float32) if total is None else total + image
        count += 1
    return None if total is None else total / count


def find_hood(average, y_from):
    """A rough guess at the row where the hood starts, for cameras not in HOOD_Y.

    The hood is in the same place in every image, so it stays sharp when
    many images are averaged while the road blurs out. This tends to land
    on the wipers, a little below the hood's front edge.
    """
    height, width = average.shape[:2]
    smooth = cv2.GaussianBlur(average, (0, 0), 2)
    change = np.abs(cv2.Sobel(smooth, cv2.CV_32F, 0, 1, ksize=3)).max(axis=2)
    lower = change[y_from:, int(0.2 * width):int(0.8 * width)]
    if np.median(lower.max(axis=0)) < 15:
        return height - 1
    hood_rows = lower.argmax(axis=0) + y_from
    return int(np.percentile(hood_rows, 10)) - 8


def camera_box(camera, image_names, average):
    height, width = average.shape[:2]
    lines = {"left": [], "right": []}
    for image_name in image_names:
        for side, lane in own_lane(read_lanes(image_name), width, height).items():
            if lane is not None and np.ptp(lane[:, 1]) > 0:
                lines[side].append(np.polyfit(lane[:, 1], lane[:, 0], 1))
    if min(len(lines["left"]), len(lines["right"])) < 10:
        return None
    left = np.median(lines["left"], axis=0)
    right = np.median(lines["right"], axis=0)
    y_meet = (right[1] - left[1]) / (left[0] - right[0])
    y_bottom = HOOD_Y.get(camera)
    if y_bottom is None:
        y_bottom = find_hood(average, int(y_meet + 0.3 * (height - y_meet)))
    y_top = y_meet + 0.2 * (y_bottom - y_meet)
    corners = np.array([
        [np.polyval(left, y_bottom), y_bottom], [np.polyval(left, y_top), y_top],
        [np.polyval(right, y_top), y_top], [np.polyval(right, y_bottom), y_bottom],
    ])
    padding = (corners[3, 0] - corners[0, 0]) * ROI_EXPANSION / 2
    corners[[0, 1], 0] -= padding
    corners[[2, 3], 0] += padding
    corners[:, 0] = np.clip(corners[:, 0], 0, width - 1)
    return np.rint(corners).astype(np.int32)


BOXES = {}
previews = []
for camera, group in images.groupby("camera"):
    sample = list(group["image"].sample(min(300, len(group)), random_state=SEED))
    average = average_image(sample)
    box = None if average is None else camera_box(camera, sample, average)
    if box is None:
        print(camera, "- images would not open or too few labels; skipping it")
        continue
    BOXES[camera] = box
    print(camera, "box:", box.tolist())
    example = next(image for image in map(load, sample) if image is not None)
    previews.append((camera, example, average.astype(np.uint8)))

figure, axes = plt.subplots(2, len(previews), figsize=(6 * len(previews), 4.8), squeeze=False)
for column, (camera, example, average) in enumerate(previews):
    for row, (picture, label) in enumerate(((example, "one image"), (average, "average of many"))):
        picture = picture.copy()
        cv2.polylines(picture, [BOXES[camera]], True, (52, 104, 235), 4)
        axes[row, column].imshow(cv2.cvtColor(picture, cv2.COLOR_BGR2RGB))
        axes[row, column].set_title(f"{camera} - {label}", fontsize=10)
        axes[row, column].axis("off")
plt.tight_layout()
plt.show()

## 4. Score every image and build the table

For each side that has a true line, the detector's line is compared with it at several
heights inside the box; the side's error is the average gap in pixels. The image's
error is its worse side.

Scene features are measured inside the box, from the image and the labels only:

- `brightness`: mean grey level
- `contrast`: spread of grey levels
- `edge_density`: share of Canny edge pixels
- `curve_px`: how far the true line bows away from a straight line
- `line_contrast`: how much brighter the true line is than the road right beside it,
  for the fainter of the two lines. CULane labels a line even where the paint is worn
  away or hidden, so this says whether there was anything to see.

In [ ]:
def true_x(lane, heights):
    """The true line's x at each height; NaN where the label doesn't reach."""
    x = np.interp(heights, lane[:, 1], lane[:, 0])
    x[(heights < lane[0, 1]) | (heights > lane[-1, 1])] = np.nan
    return x


def side_error(line, lane, heights):
    if lane is None:
        return None
    truth = true_x(lane, heights)
    if np.isnan(truth).all():
        return None
    if line is None:
        return float(ERROR_CAP)
    gap = np.nanmean(np.abs(np.polyval(line, heights) - truth))
    return float(min(gap, ERROR_CAP))


def bow(lane, y_top):
    """Largest gap in pixels between the true line and a straight line through it."""
    if lane is None:
        return np.nan
    part = lane[lane[:, 1] >= y_top]
    if len(part) < 3:
        return 0.0
    straight = np.polyfit(part[:, 1], part[:, 0], 1)
    return float(np.max(np.abs(part[:, 0] - np.polyval(straight, part[:, 1]))))


def line_contrast(gray, lane, y_top, y_bottom):
    """Grey levels by which the true line is brighter than the road beside it."""
    if lane is None:
        return np.nan
    ys = np.arange(int(y_top), int(y_bottom), 4, dtype=float)
    xs = np.interp(ys, lane[:, 1], lane[:, 0])
    usable = (
        (ys >= lane[0, 1]) & (ys <= lane[-1, 1]) & (xs >= 40) & (xs < gray.shape[1] - 40)
    )
    if usable.sum() < 5:
        return np.nan
    ys, xs = ys[usable].astype(int), np.rint(xs[usable]).astype(int)
    # Labels are a few pixels off, so take the brightest pixel near the line.
    on_line = np.max([gray[ys, xs + shift] for shift in range(-10, 11)], axis=0)
    beside = np.median(
        [gray[ys, xs + shift] for shift in (-30, -25, -20, 20, 25, 30)], axis=0
    )
    return float(np.median(on_line.astype(float) - beside))


def score_image(item):
    image = load(item.image)
    if image is None or item.camera not in BOXES:
        return "image would not open"
    lanes = read_lanes(item.image)
    box = BOXES[item.camera]
    height, width = image.shape[:2]
    truth = own_lane(lanes, width, height)
    heights = np.linspace(box[:, 1].min(), box[:, 1].max(), SCORE_HEIGHTS)

    lines = dict(zip(("left", "right"), detect_lanes(image, box)))
    errors = [side_error(lines[side], truth[side], heights) for side in lines]
    errors = [error for error in errors if error is not None]
    if not errors:
        return "no labelled lane"

    inside = box_mask(image.shape, box) > 0
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    smooth = cv2.GaussianBlur(gray, (5, 5), 0)
    faintest = np.array([
        line_contrast(smooth, lane, heights[0], heights[-1]) for lane in truth.values()
    ])
    return {
        "image": item.image,
        "category": item.category,
        "camera": item.camera,
        "brightness": float(gray[inside].mean()),
        "contrast": float(gray[inside].std()),
        "edge_density": float((lane_edges(image)[inside] > 0).mean()),
        "curve_px": float(np.nanmax([bow(lane, heights[0]) for lane in truth.values()])),
        "line_contrast": np.nan if np.isnan(faintest).all() else float(np.nanmin(faintest)),
        "error": max(errors),
        "missed": any(lines[side] is None and truth[side] is not None for side in lines),
        "passed": max(errors) <= PASS_PX,
    }


with ThreadPoolExecutor(max_workers=os.cpu_count() or 2) as pool:
    outcomes = list(pool.map(score_image, images.itertuples(index=False)))

table = pd.DataFrame([outcome for outcome in outcomes if isinstance(outcome, dict)])
skipped = pd.Series([o for o in outcomes if isinstance(o, str)]).value_counts()
print(f"Scored {len(table)} of {len(images)} images.")
if len(skipped):
    print("Left out:", skipped.to_dict())
table.to_csv("lane_results.csv", index=False)

summary = table.groupby("category").agg(
    images=("passed", "size"),
    pass_rate=("passed", "mean"),
    median_error=("error", "median"),
    missed_a_line=("missed", "mean"),
).sort_values("pass_rate", ascending=False)
print(f"\nOverall pass rate: {table['passed'].mean():.1%}")
display(summary.round(2))

# How much of the failing is simply "there was no visible line to find"?
visible = table["line_contrast"] >= VISIBLE_CONTRAST
print(f"Both true lines clearly visible in {visible.mean():.0%} of images.")
print(f"  pass rate when they are:     {table.loc[visible, 'passed'].mean():.1%}")
print(f"  pass rate when they are not: {table.loc[~visible, 'passed'].mean():.1%}")
table.head()

## 5. Which scene features explain the failures?

A random forest predicts pass/fail from the five scene features only. Nothing that
comes from the detector goes in (not `missed`, not pixel counts), and neither does the
category label. Importance is measured on held-out images: how much the score drops
when one feature's values are shuffled.

In [ ]:
FEATURES = ["brightness", "contrast", "edge_density", "curve_px", "line_contrast"]

X = table[FEATURES].fillna(table[FEATURES].median())
y = table["passed"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)
forest = RandomForestClassifier(
    n_estimators=300, min_samples_leaf=5, class_weight="balanced",
    random_state=SEED, n_jobs=-1,
).fit(X_train, y_train)

predicted = forest.predict(X_test)
always_common = max(y_test.mean(), 1 - y_test.mean())
print(f"Accuracy:          {accuracy_score(y_test, predicted):.1%}"
      f"  (always guessing the commoner outcome: {always_common:.1%})")
print(f"Balanced accuracy: {balanced_accuracy_score(y_test, predicted):.1%}  (chance: 50%)")

shuffled = permutation_importance(
    forest, X_test, y_test, scoring="balanced_accuracy", n_repeats=20, random_state=SEED
)
importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": shuffled.importances_mean,
    "spread": shuffled.importances_std,
}).sort_values("importance")
importance.sort_values("importance", ascending=False).round(3)

## 6. Charts

In [ ]:
BLUE, INK, MUTED, GRID = "#2a78d6", "#0b0b0b", "#52514e", "#e4e3df"
TRUE_BGR, FOUND_BGR = (122, 175, 27), (52, 104, 235)  # aqua, orange
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": GRID, "axes.labelcolor": MUTED, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.titlesize": 12,
    "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "axes.axisbelow": True,
})

# Pass rate per category
rates = summary.sort_values("pass_rate")
figure, axis = plt.subplots(figsize=(7, 0.45 * len(rates) + 1.2))
bars = axis.barh(
    [f"{name}  (n={count})" for name, count in zip(rates.index, rates["images"])],
    rates["pass_rate"] * 100, color=BLUE, height=0.6,
)
axis.bar_label(bars, fmt="%.0f%%", padding=4, color=INK)
axis.set_xlim(0, 108)
axis.set_xlabel(f"Images passing (worse line within {PASS_PX} px), %")
axis.set_title("Pass rate per category")
axis.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()


def error_chart(feature, label, x_max=None):
    """Top: pass rate in ten equal-sized groups. Bottom: every image's error."""
    name = label.split(" (")[0].lower()
    figure, (rate_axis, error_axis) = plt.subplots(
        2, 1, figsize=(7, 5.6), sharex=True, gridspec_kw={"height_ratios": [1, 1.3]}
    )
    groups = pd.qcut(table[feature], 10, duplicates="drop")
    grouped = table.groupby(groups, observed=True).agg(
        x=(feature, "median"), rate=("passed", "mean")
    )
    rate_axis.plot(grouped["x"], grouped["rate"] * 100, color=BLUE, linewidth=2,
                   marker="o", markersize=6)
    rate_axis.set_ylim(0, max(50, grouped["rate"].max() * 110))
    rate_axis.set_ylabel("Images passing, %")
    rate_axis.set_title(f"Pass rate and error vs {name}")

    error_axis.scatter(table[feature], table["error"], s=8, color=BLUE, alpha=0.25,
                       linewidths=0)
    error_axis.axhline(PASS_PX, color=MUTED, linewidth=1, linestyle="--")
    error_axis.annotate(f"pass limit, {PASS_PX} px", (1, PASS_PX),
                        xycoords=("axes fraction", "data"), xytext=(-4, 4),
                        textcoords="offset points", ha="right", color=MUTED, fontsize=9)
    error_axis.set_ylim(0, ERROR_CAP * 1.05)
    error_axis.set_ylabel(f"Error, px (capped at {ERROR_CAP})")
    if x_max is not None:
        beyond = (table[feature] > x_max).mean()
        error_axis.set_xlim(left=min(0, table[feature].min()) - x_max * 0.02, right=x_max)
        label += f"; {beyond:.0%} of images are beyond {x_max} and not shown"
    error_axis.set_xlabel(label)
    plt.tight_layout()
    plt.show()


error_chart("brightness", "Brightness (mean grey level in the box, 0-255)")
error_chart("curve_px", "Curvature (true line's bow from straight, px)", x_max=60)
error_chart("line_contrast", "Line contrast (true line vs road beside it, grey levels)")

# Feature importance
figure, axis = plt.subplots(figsize=(7, 0.5 * len(importance) + 1.2))
axis.barh(importance["feature"], importance["importance"], xerr=importance["spread"],
          color=BLUE, height=0.6, ecolor=MUTED, capsize=3)
axis.axvline(0, color=MUTED, linewidth=1)
axis.set_xlabel("Drop in balanced accuracy when the feature is shuffled")
axis.set_title("What the forest relies on to predict pass/fail")
axis.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()


# The worst failures: images where the detector drew lines that are wrong
def draw_line(image, line, y_top, y_bottom, color):
    ys = np.arange(int(y_top), int(y_bottom))
    points = np.column_stack((np.polyval(line, ys), ys)).astype(np.int32)
    cv2.polylines(image, [points], False, color, 6)


drew_something = table[~table["missed"]]
worst = (drew_something if len(drew_something) else table).nlargest(12, "error")
figure, axes = plt.subplots(4, 3, figsize=(15, 8.2))
for axis in axes.ravel():
    axis.axis("off")
for axis, item in zip(axes.ravel(), worst.itertuples()):
    image = load(item.image)
    box = BOXES[item.camera]
    height, width = image.shape[:2]
    for lane in own_lane(read_lanes(item.image), width, height).values():
        if lane is not None:
            cv2.polylines(image, [lane.astype(np.int32)], False, TRUE_BGR, 6)
    for line in detect_lanes(load(item.image), box):
        if line is not None:
            draw_line(image, line, box[:, 1].min(), box[:, 1].max(), FOUND_BGR)
    axis.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axis.set_title(f"{item.category} · error {item.error:.0f} px", fontsize=10, loc="left")
figure.suptitle("Worst failures  (green = true line, orange = detector)",
                x=0.01, ha="left", fontweight="bold")
plt.tight_layout()
plt.show()